In [ ]:
import pandas as pd
import os
import glob
from datetime import datetime

# Define the folder path
folder_path = '/Volumes/T7/bid-price-data-sorted-feather-B1'

# Get all feather files in the folder
feather_files = glob.glob(os.path.join(folder_path, '*.feather'))
print(f"Found {len(feather_files)} feather files")

# List to store results
hpr1_results = []

# Process each file
for file_path in feather_files:
    print(f"Processing {os.path.basename(file_path)}...")
    
    try:
        # Read the feather file
        df = pd.read_feather(file_path)
        
        # Check if DUID and SETTLEMENTDATE columns exist
        if 'DUID' not in df.columns or 'SETTLEMENTDATE' not in df.columns:
            print(f"  Missing required columns in {os.path.basename(file_path)}")
            continue
        
        # Ensure SETTLEMENTDATE is datetime
        if not pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
            df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'], errors='coerce')
        
        # Filter for dates after 2017 and DUID containing HPR1
        filtered_df = df[(df['SETTLEMENTDATE'] >= '2018-01-01') & 
                          (df['DUID'].str.contains('H', case=False, na=False))]
        
        # If we found matches, store the information
        if not filtered_df.empty:
            file_info = {
                'file_name': os.path.basename(file_path),
                'matches_count': len(filtered_df),
                'unique_duids': filtered_df['DUID'].unique().tolist(),
                'date_range': (filtered_df['SETTLEMENTDATE'].min(), 
                              filtered_df['SETTLEMENTDATE'].max())
            }
            hpr1_results.append(file_info)
            
            print(f"  Found {len(filtered_df)} matches with DUID: {filtered_df['DUID'].unique().tolist()}")
        else:
            print(f"  No matches found")
            
    except Exception as e:
        print(f"  Error processing {os.path.basename(file_path)}: {str(e)}")

# Display summary of results
print("\n=== SUMMARY ===")
print(f"Found HPR1 entries in {len(hpr1_results)} files")

for result in hpr1_results:
    print(f"\nFile: {result['file_name']}")
    print(f"Matches: {result['matches_count']}")
    print(f"DUIDs: {result['unique_duids']}")
    print(f"Date range: {result['date_range'][0]} to {result['date_range'][1]}")

# Optional: Combine all matching data
if hpr1_results:
    print("\nCombining all matching data...")
    combined_data = []
    
    for file_info in hpr1_results:
        df = pd.read_feather(os.path.join(folder_path, file_info['file_name']))
        
        # Ensure SETTLEMENTDATE is datetime
        if not pd.api.types.is_datetime64_any_dtype(df['SETTLEMENTDATE']):
            df['SETTLEMENTDATE'] = pd.to_datetime(df['SETTLEMENTDATE'], errors='coerce')
        
        # Filter for dates after 2017 and DUID containing HPR1
        filtered_df = df[(df['SETTLEMENTDATE'] >= '2018-01-01') & 
                          (df['DUID'].str.contains('H', case=False, na=False))]
        
        combined_data.append(filtered_df)
    
    # Combine all filtered data
    if combined_data:
        all_hpr1_data = pd.concat(combined_data, ignore_index=True)
        print(f"Combined data shape: {all_hpr1_data.shape}")
        
        # Save combined data to a new file
        output_path = os.path.join(folder_path, 'hpr1_combined_data.feather')
        all_hpr1_data.to_feather(output_path)
        print(f"Combined data saved to {output_path}")